In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSV files (they are in the current working directory)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Helper: convert epoch seconds series to min/max ISO strings in UTC+8
def epoch_to_tz_iso(epoch_series):
    if ('timestamp' not in epoch_series.index) and isinstance(epoch_series, pd.Series):
        # fallback if a Series of timestamps passed directly
        s = epoch_series
    elif isinstance(epoch_series, pd.Series):
        s = epoch_series
    else:
        return (None, None)
    if s.dropna().empty:
        return (None, None)
    dt = pd.to_datetime(s.dropna(), unit='s', utc=True).dt.tz_convert(tz)
    return dt.min().isoformat(), dt.max().isoformat()

# Generic summarizer (robust column handling)
def summarize_file(df, name_col=None, sample_col_name='name'):
    total_rows = len(df)
    min_ts, max_ts = (None, None)
    if 'timestamp' in df.columns:
        min_ts, max_ts = epoch_to_tz_iso(df['timestamp'])
    # cmdb top counts (robust)
    if 'cmdb_id' in df.columns:
        cmdb_counts = df['cmdb_id'].value_counts().reset_index()
        cmdb_counts.columns = ['cmdb_id', 'count']
        cmdb_top = cmdb_counts.head(50)
        unique_cmdb_count = df['cmdb_id'].nunique()
    else:
        cmdb_top = pd.DataFrame(columns=['cmdb_id','count'])
        unique_cmdb_count = 0
    # name/message top counts (robust)
    if name_col and name_col in df.columns:
        name_counts = df[name_col].value_counts().reset_index()
        name_counts.columns = [sample_col_name, 'count']
        name_top = name_counts.head(50)
        unique_names_count = name_counts.shape[0]
    else:
        name_top = pd.DataFrame(columns=[sample_col_name, 'count'])
        unique_names_count = 0
    summary = pd.DataFrame([{
        'total_rows': total_rows,
        'min_timestamp_UTC+8': min_ts,
        'max_timestamp_UTC+8': max_ts,
        'unique_cmdb_count': unique_cmdb_count,
        'unique_names_count': unique_names_count
    }])
    return summary, cmdb_top, name_top

# Summarize each file
metric_summary, metric_cmdb_top, metric_kpi_top = summarize_file(metric_df, name_col='kpi_name', sample_col_name='kpi_name')
trace_summary, trace_cmdb_top, trace_trace_top = summarize_file(trace_df, name_col='trace_name', sample_col_name='trace_name')
log_summary, log_cmdb_top, log_logname_top = summarize_file(log_df, name_col='log_name', sample_col_name='log_name')
error_summary, error_cmdb_top, error_msg_top = summarize_file(error_df, name_col='message', sample_col_name='message')

# Keep outputs compact (top 50 as requested). Show selected columns only.
metric_summary, metric_cmdb_top[['cmdb_id','count']].head(50), metric_kpi_top[['kpi_name','count']].head(50), \
trace_summary, trace_cmdb_top[['cmdb_id','count']].head(50), trace_trace_top[['trace_name','count']].head(50), \
log_summary, log_cmdb_top[['cmdb_id','count']].head(50), log_logname_top[['log_name','count']].head(50), \
error_summary, error_cmdb_top[['cmdb_id','count']].head(50), error_msg_top[['message','count']].head(50)

```
Out[1]:
```
```python
summary = (
    "Summary of loaded telemetry files:\n\n"
    "1) metric.csv\n"
    "- Total rows: 29,055\n"
    "- Time range (UTC+8): 2021-03-24 14:00:00 → 2021-03-24 14:30:00\n"
    "- Unique cmdb_ids: 26 (top 5 by row count: Mysql01=4726, Mysql02=4644, Tomcat03=2049, Tomcat02=2013, Tomcat04=1976)\n"
    "- Unique KPI names: 356 (top 5: app.mrt=341, app.cnt=341, app.sr=341, app.rr=341, OSLinux-..._TCP-FIN-WAIT=339)\n\n"
    "2) trace.csv\n"
    "- Total rows: 6,768\n"
    "- Time range (UTC+8): 2021-03-24 14:00:00 → 2021-03-24 14:30:00\n"
    "- Unique cmdb_ids: 12 (top 6: MG02=1134, MG01=1134, IG01=468, IG02=465, Tomcat04=459, Tomcat02=453)\n"
    "- Unique trace names: 72 (top 5: trace.self.duration_mean=360, trace.self.duration_p95=360, trace.self.row_count=360, trace.to_MG01.duration_p95=226, trace.to_MG01.row_count=226)\n\n"
    "3) log.csv\n"
    "- Total rows: 494\n"
    "- Time range (UTC+8): 2021-03-24 14:00:00 → 2021-03-24 14:30:00\n"
    "- Unique cmdb_ids: 7 (top 7: Tomcat04=83, Tomcat02=81, Tomcat03=77, Tomcat01=73, apache01=62, apache02=62, Mysql01=56)\n"
    "- Unique log names: 3 (log.error_count=208, log.row_count=208, log.gc_pause_max=78)\n\n"
    "4) error_logs.csv\n"
    "- Total rows: 118\n"
    "- Time range (UTC+8): 2021-03-24 14:20:09 → 2021-03-24 14:26:50\n"
    "- Unique cmdb_ids: 6 (top 6: Tomcat03=31, Tomcat04=23, Tomcat02=20, Tomcat01=20, apache02=14, apache01=10)\n"
    "- Unique messages: 118 (messages are mostly unique; top message counts are 1)\n\n"
    "Overall: metric and trace data are dense across the 30-minute window (many KPI/trace types); logs are smaller and error logs are short but contain mostly unique message entries concentrated in a ~7-minute window."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows        min_timestamp_UTC+8        max_timestamp_UTC+8  unique_cmdb_count  unique_names_count
0       29055  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00                 26                 356,          cmdb_id  count
0        Mysql01   4726
1        Mysql02   4644
2       Tomcat03   2049
3       Tomcat02   2013
4       Tomcat04   1976
..           ...    ...
21  ServiceTest5    124
22  ServiceTest4    124
23  ServiceTest3    124
24  ServiceTest9    124
25      dockerA2    114

[26 rows x 2 columns],                                              kpi_name  count
0                                             app.mrt    341
1                                             app.cnt    341
2                                              app.sr    341
3                                              app.rr    341
4        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    339
..                                                ...    ...
45  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...    134
46                    OSLinux-CPU_CPU-2_SingleCpuUtil    132
47                    OSLinux-CPU_CPU-0_SingleCpuUtil    129
48                    OSLinux-CPU_CPU-1_SingleCpuidle    128
49                    OSLinux-CPU_CPU-3_SingleCpuidle    128

[50 rows x 2 columns],    total_rows        min_timestamp_UTC+8        max_timestamp_UTC+8  unique_cmdb_count  unique_names_count
0        6768  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00                 12                  72,      cmdb_id  count
0       MG02   1134
1       MG01   1134
2       IG01    468
3       IG02    465
4   Tomcat04    459
5   Tomcat02    453
6   Tomcat03    453
7   dockerA2    453
8   Tomcat01    450
9   dockerB1    447
10  dockerB2    441
11  dockerA1    411,                          trace_name  count
0          trace.self.duration_mean    360
1           trace.self.duration_p95    360
2              trace.self.row_count    360
3        trace.to_MG01.duration_p95    226
4           trace.to_MG01.row_count    226
..                              ...    ...
45   trace.to_dockerB2.duration_p95     58
46  trace.to_dockerB1.duration_mean     58
47   trace.to_dockerB1.duration_p95     58
48      trace.to_dockerB1.row_count     58
49   trace.to_dockerA2.duration_p95     58

[50 rows x 2 columns],    total_rows        min_timestamp_UTC+8        max_timestamp_UTC+8  unique_cmdb_count  unique_names_count
0         494  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00                  7                   3,     cmdb_id  count
0  Tomcat04     83
1  Tomcat02     81
2  Tomcat03     77
3  Tomcat01     73
4  apache01     62
5  apache02     62
6   Mysql01     56,            log_name  count
0   log.error_count    208
1     log.row_count    208
2  log.gc_pause_max     78,    total_rows        min_timestamp_UTC+8        max_timestamp_UTC+8  unique_cmdb_count  unique_names_count
0         118  2021-03-24T14:20:09+08:00  2021-03-24T14:26:50+08:00                  6                 118,     cmdb_id  count
0  Tomcat03     31
1  Tomcat04     23
2  Tomcat02     20
3  Tomcat01     20
4  apache02     14
5  apache01     10,                                               message  count
0   IPAddress "POST /UOCP/person/ServiceTest1.json...      1
1   IPAddress "POST /UOCP/person/ServiceTest3.json...      1
2   IPAddress "POST /UOCP/person/ServiceTest1.json...      1
3   IPAddress "POST /UOCP/base/ServiceTest8.json H...      1
4   IG01 POST /UOCP/person/ServiceTest9.json HTTP/...      1
..                                                ...    ...
45  IG02 POST /UOCP/person/ServiceTest5.json HTTP/...      1
46  IG01 POST /UOCP/person/ServiceTest9.json HTTP/...      1
47  IG02 POST /UOCP/person/ServiceTest1.json HTTP/...      1
48  IG02 POST /UOCP/person/ServiceTest10.json HTTP...      1
49  IG01 POST /UOCP/person/ServiceTest5.json HTTP/...      1

[50 rows x 2 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Reuse existing metric_df and tz from earlier state
components = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
              'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

# Filter metric rows to the requested components
filtered = metric_df[metric_df['cmdb_id'].isin(components)].copy()

# Group and compute required aggregates: count, global P95 of value, min and max timestamp
grouped = (
    filtered
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count=('value', 'size'),
        p95_value=('value', lambda s: s.dropna().quantile(0.95)),
        min_ts=('timestamp', 'min'),
        max_ts=('timestamp', 'max')
    )
)

# Convert epoch seconds to UTC+8 ISO strings for min/max timestamps
grouped['min_timestamp'] = grouped['min_ts'].apply(
    lambda x: pd.to_datetime(x, unit='s', utc=True).tz_convert(tz).isoformat() if pd.notnull(x) else None
)
grouped['max_timestamp'] = grouped['max_ts'].apply(
    lambda x: pd.to_datetime(x, unit='s', utc=True).tz_convert(tz).isoformat() if pd.notnull(x) else None
)

# Select required columns, sort by count desc, and limit to top 20 groups
top20_groups = (
    grouped[['cmdb_id', 'kpi_name', 'count', 'p95_value', 'min_timestamp', 'max_timestamp']]
    .sort_values(by='count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

top20_groups

```
Out[2]:
```
```python
# Reuse existing top20_groups from previous computation and tz
# Build a concise plain-English summary string based on the actual DataFrame content.

total_groups = top20_groups.shape[0]
count_values = sorted(top20_groups['count'].unique())
cmdb_counts = top20_groups['cmdb_id'].value_counts().head(10)
time_min = top20_groups['min_timestamp'].min()
time_max = top20_groups['max_timestamp'].max()
top_p95_rows = top20_groups.sort_values('p95_value', ascending=False).head(3)

top_p95_list = [
    f"{i+1}. {row['cmdb_id']} / {row['kpi_name']} -> p95={row['p95_value']:.6g}"
    for i, row in top_p95_rows.reset_index(drop=True).iterrows()
]

summary = (
    f"Top 20 (cmdb_id, kpi_name) groups summary:\n"
    f"- Groups returned: {total_groups}\n"
    f"- Count values present: {count_values} (per-group 'count' column)\n"
    f"- Components represented (top counts among these 20 groups):\n"
    f"  {', '.join([f'{idx}={cnt}' for idx, cnt in cmdb_counts.items()])}\n"
    f"- Time range across these groups (UTC+8): {time_min} → {time_max}\n"
    f"- Highest P95 values (top 3):\n  " + "\n  ".join(top_p95_list) + "\n"
    f"\nObservations:\n"
    f"- All returned groups have the same sample count ({count_values[0]}), indicating uniform sampling across the 30-minute window.\n"
    f"- Many top groups are from Tomcat04 (dominant in this top-20), with several Tomcat session/thread/request KPIs among the highest P95 values.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count     p95_value              min_timestamp              max_timestamp
0    Mysql01           Mysql-MySQL_3306_GetResponseTimeOfMysqld     31  0.000000e+00  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00
1   Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  3.895000e+02  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00
2       IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31  7.400000e+01  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00
3   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00
4   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00
5   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00
6   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  3.830000e+02  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00
7   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  1.685000e+02  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00
8   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31  5.000000e+02  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00
9   Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31  0.000000e+00  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00
10  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...     31  0.000000e+00  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00
11  Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter     31  0.000000e+00  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00
12  Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter     31  0.000000e+00  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00
13  Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31  0.000000e+00  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00
14  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00
15  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00
16  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00
17  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31  4.110000e+02  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00
18  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31  3.802840e+05  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00
19  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  1.150253e+09  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00```
```

In [ ]:
# In[3]:

# Reuse variables from previous state: metric_df, components, tz
import pandas as pd
import numpy as np
from pytz import timezone

# Ensure components and tz exist (redefine if necessary)
components = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
              'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']
tz = timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, kpi_name) using all available rows for these components
global_p95 = (
    metric_df[metric_df['cmdb_id'].isin(components)]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(p95_value=('value', lambda s: s.dropna().quantile(0.95)))
)

# 2) Filter metric rows to requested components and the UTC+8 time window 2021-03-24 14:00:00 → 14:30:00
start_local = pd.Timestamp('2021-03-24 14:00:00').tz_localize(tz)
end_local = pd.Timestamp('2021-03-24 14:30:00').tz_localize(tz)

df_filtered = metric_df[metric_df['cmdb_id'].isin(components)].copy()
# convert epoch -> tz-aware datetime (UTC->convert to Asia/Shanghai)
df_filtered['dt'] = pd.to_datetime(df_filtered['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
# filter by local time window (inclusive)
df_window = df_filtered[(df_filtered['dt'] >= start_local) & (df_filtered['dt'] <= end_local)].copy()

# 3) Attach global p95 to windowed rows
df_window = df_window.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# 4) Aggregate per (cmdb_id, kpi_name)
def agg_group(g):
    p95 = g['p95_value'].iloc[0]
    # handle NaN p95: treat as no anomalies
    if pd.isna(p95):
        anomaly_count = 0
        earliest = pd.NaT
    else:
        anomaly_mask = g['value'] >= p95
        anomaly_count = int(anomaly_mask.sum())
        earliest = g.loc[anomaly_mask, 'dt'].min() if anomaly_count > 0 else pd.NaT
    max_value = g['value'].max() if not g['value'].empty else np.nan
    # max_over_p95 handling
    if pd.isna(p95) or p95 == 0:
        max_over = None
    else:
        max_over = max_value / p95
    return pd.Series({
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest.isoformat() if pd.notnull(earliest) else None,
        'max_value': max_value,
        'p95_value': p95,
        'max_over_p95': max_over
    })

agg = (
    df_window
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .apply(lambda g: agg_group(g))
    .reset_index()
)

# 5) Return top 20 groups ordered by anomaly_count descending
result_top20 = (
    agg[['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value','p95_value','max_over_p95']]
    .sort_values(by='anomaly_count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

result_top20

```
Out[3]:
```
```python
# Build a concise plain-English summary from the previously computed result_top20
df = result_top20

total_rows = df.shape[0]
anomaly_counts = sorted(df['anomaly_count'].unique())
components_freq = df['cmdb_id'].value_counts().head(10).to_dict()
p95_zero_count = int((df['p95_value'] == 0).sum())
earliest_ts = df['earliest_anomaly_timestamp'].dropna().unique()
highest_max_over = df['max_over_p95'].dropna()
highest_max_over_val = float(highest_max_over.max()) if not highest_max_over.empty else None

summary = (
    f"Anomaly summary for top {total_rows} groups (30-min window, UTC+8):\n"
    f"- All returned groups have anomaly_count values: {anomaly_counts} (most groups show 31 samples flagged).\n"
    f"- Earliest anomaly timestamps observed (samples): {list(earliest_ts)} (mostly 2021-03-24T14:00:00+08:00).\n"
    f"- Components represented (top counts in these {total_rows} rows): {components_freq}.\n"
    f"- p95 == 0 occurred in {p95_zero_count} groups; for these groups max_over_p95 is unset/null.\n"
    f"- Many groups have max_value equal to p95_value (max_over_p95 = 1.0). Highest observed max_over_p95 in this top-20: {highest_max_over_val}.\n\n"
    f"Key observations:\n"
    f"- The top groups are dominated by Tomcat and IG JVM metrics (session/thread/request/memory KPIs).\n"
    f"- The anomaly_count being equal to the full sample count (31) suggests values met or exceeded their global P95 at every minute in the 30-minute window for these groups.\n"
    f"- Where p95 is zero, ratio-based severity is unavailable and should be treated separately.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp     max_value     p95_value  max_over_p95
0       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed             31  2021-03-24T14:00:00+08:00  5.848308e+07  5.848308e+07           1.0
1       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31  2021-03-24T14:00:00+08:00  2.040136e+09  2.040136e+09           1.0
2   Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...             31  2021-03-24T14:00:00+08:00  5.000000e+02  5.000000e+02           1.0
3   Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31  2021-03-24T14:00:00+08:00  4.016560e+05  4.016560e+05           1.0
4   Tomcat03     Tomcat-Sessions_7441--_SESSIONRejectedSessions             31  2021-03-24T14:00:00+08:00  0.000000e+00  0.000000e+00           NaN
5   Tomcat03        Tomcat-Sessions_7441--_SESSIONActiveCounter             31  2021-03-24T14:00:00+08:00  0.000000e+00  0.000000e+00           NaN
6   Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31  2021-03-24T14:00:00+08:00  0.000000e+00  0.000000e+00           NaN
7       MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31  2021-03-24T14:00:00+08:00  1.000000e+00  0.000000e+00           NaN
8   Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31  2021-03-24T14:00:00+08:00  3.978080e+05  3.978080e+05           1.0
9   Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31  2021-03-24T14:00:00+08:00  0.000000e+00  0.000000e+00           NaN
10  Tomcat01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31  2021-03-24T14:00:00+08:00  3.000000e-01  3.000000e-01           1.0
11  Tomcat01     Tomcat-Sessions_7441--_SESSIONRejectedSessions             31  2021-03-24T14:00:00+08:00  0.000000e+00  0.000000e+00           NaN
12  Tomcat01     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter             31  2021-03-24T14:00:00+08:00  0.000000e+00  0.000000e+00           NaN
13  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31  2021-03-24T14:00:00+08:00  0.000000e+00  0.000000e+00           NaN
14  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31  2021-03-24T14:00:00+08:00  0.000000e+00  0.000000e+00           NaN
15  Tomcat03     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter             31  2021-03-24T14:00:00+08:00  0.000000e+00  0.000000e+00           NaN
16  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31  2021-03-24T14:00:00+08:00  0.000000e+00  0.000000e+00           NaN
17  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31  2021-03-24T14:00:00+08:00  0.000000e+00  0.000000e+00           NaN
18  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31  2021-03-24T14:00:00+08:00  3.906190e+05  3.906190e+05           1.0
19  Tomcat01        Tomcat-Sessions_7441--_SESSIONActiveCounter             31  2021-03-24T14:00:00+08:00  0.000000e+00  0.000000e+00           NaN```
```

In [ ]:
# In[4]:

# Reuse previously computed 'agg' DataFrame (per-(cmdb_id,kpi_name) results)
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure agg exists; it should have columns:
# ['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value','p95_value','max_over_p95']
df_anom = agg.copy()

# Normalize types: anomaly_count -> int, max_over_p95 -> numeric (None -> NaN)
df_anom['anomaly_count'] = df_anom['anomaly_count'].fillna(0).astype(int)
df_anom['max_over_p95'] = pd.to_numeric(df_anom['max_over_p95'], errors='coerce')

# Parse earliest_anomaly_timestamp into tz-aware datetime (None/NaT preserved)
df_anom['earliest_dt'] = pd.to_datetime(df_anom['earliest_anomaly_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)

# Aggregation functions per cmdb_id
def agg_per_cmdb(g):
    sum_anomaly = int(g['anomaly_count'].sum())
    num_kpis_flagged = int((g['anomaly_count'] > 0).sum())
    # earliest anomaly timestamp (min) in ISO format or None
    if g['earliest_dt'].dropna().empty:
        earliest_iso = None
    else:
        earliest_iso = g['earliest_dt'].min().isoformat()
    # peak severity: max of max_over_p95 ignoring NaN
    if g['max_over_p95'].dropna().empty:
        peak_sev = None
    else:
        peak_sev = float(g['max_over_p95'].max())
    # top 3 kpis ranking by anomaly_count desc then max_over_p95 desc (NaN treated as smallest)
    ranked = g.assign(_rank_over=g['max_over_p95'].fillna(-np.inf)) \
              .sort_values(['anomaly_count', '_rank_over'], ascending=[False, False])
    top3 = ranked['kpi_name'].head(3).astype(str).tolist()
    top3_str = ", ".join(top3)
    return pd.Series({
        'sum_anomaly_count': sum_anomaly,
        'num_kpis_flagged': num_kpis_flagged,
        'earliest_anomaly_timestamp': earliest_iso,
        'peak_severity': peak_sev,
        'top_3_kpis': top3_str
    })

cmdb_agg = df_anom.groupby('cmdb_id', as_index=False).apply(lambda g: agg_per_cmdb(g)).reset_index()

# Order by sum_anomaly_count desc and limit to top 20 rows
cmdb_agg_top20 = cmdb_agg.sort_values('sum_anomaly_count', ascending=False).head(20).reset_index(drop=True)

# Keep columns in requested order
cmdb_agg_top20 = cmdb_agg_top20[['cmdb_id','sum_anomaly_count','num_kpis_flagged','earliest_anomaly_timestamp','peak_severity','top_3_kpis']]

cmdb_agg_top20

```
Out[4]:
```
Summary of aggregated anomaly results by component (top rows):

- Top affected components by total anomaly count:
  1. Mysql01 — sum_anomaly_count=2356 across 224 KPIs flagged; earliest anomaly 2021-03-24T14:00:00+08:00; peak_severity=1.8286. Top KPIs include Mysql-MySQL_3306_GetConnectedStateOfMysqld and related MySQL KPIs.
  2. Mysql02 — sum_anomaly_count=2318 across 222 KPIs; earliest 2021-03-24T14:00:00+08:00; peak_severity=1.8182. Top KPIs similar to Mysql01.
  3. Tomcat03 — sum_anomaly_count=1164 across 123 KPIs; earliest 2021-03-24T14:00:00+08:00; peak_severity=3.7044. Top KPIs include OS/process and Tomcat process/request metrics.
  4. Tomcat04 — sum_anomaly_count=1123 across 122 KPIs; earliest 14:00; peak_severity=2.2727.
  5. Redis02 — sum_anomaly_count=1077 across 111 KPIs; earliest 14:00; peak_severity=3.3333.
  6. Tomcat02 — sum_anomaly_count=1073 across 122 KPIs; peak_severity=2.4164.
  7. Tomcat01 — sum_anomaly_count=1063 across 119 KPIs; peak_severity=4.0.
  8. Redis01 — sum_anomaly_count=969 across 115 KPIs; peak_severity=1.8182.
  9. MG01 — sum_anomaly_count=822 across 104 KPIs; peak_severity=3.4286.
 10. MG02 — sum_anomaly_count=821 across 106 KPIs; peak_severity=3.2602.
 11. IG01 — sum_anomaly_count=492 across 88 KPIs; peak_severity=2.1374.
 12. IG02 — sum_anomaly_count=469 across 86 KPIs; peak_severity=1.9684.
 13. apache02 — sum_anomaly_count=451 across 82 KPIs; peak_severity=2.0091.
 14. apache01 — sum_anomaly_count=425 across 80 KPIs; peak_severity=1.6882.

Key observations:
- Anomalies begin at the start of the 30‑minute window (most earliest timestamps are 2021-03-24T14:00:00+08:00).
- MySQL servers (Mysql01/Mysql02) show the highest total anomaly counts and the largest number of distinct KPIs flagged — these are the most broadly impacted.
- Several Tomcat and Redis nodes show high peak severity (peak_severity > 3), indicating individual KPI values exceeded their P95 by multiple times.
- Many top KPIs driving alerts are OS/process, JVM, Tomcat request/thread/session, and DB (MySQL) related metrics.

Suggested next steps:
- Prioritize investigation on Mysql01 and Mysql02 for widespread KPI anomalies, then Tomcat (03/04/02/01) and Redis nodes for high-severity spikes.
- Inspect the top KPIs listed per component (connection/state/response-time for MySQL; thread/request/MaxTime for Tomcat; process/CPU/memory for OS-level metrics) to find root causes.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  sum_anomaly_count  num_kpis_flagged earliest_anomaly_timestamp  peak_severity                                         top_3_kpis
0    Mysql01               2356               224  2021-03-24T14:00:00+08:00       1.828571  Mysql-MySQL_3306_GetConnectedStateOfMysqld, My...
1    Mysql02               2318               222  2021-03-24T14:00:00+08:00       1.818182  Mysql-MySQL_3306_GetConnectedStateOfMysqld, My...
2   Tomcat03               1164               123  2021-03-24T14:00:00+08:00       3.704434  OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies,...
3   Tomcat04               1123               122  2021-03-24T14:00:00+08:00       2.272727  OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies,...
4    Redis02               1077               111  2021-03-24T14:00:00+08:00       3.333333  OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount, OS...
5   Tomcat02               1073               122  2021-03-24T14:00:00+08:00       2.416423  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...
6   Tomcat01               1063               119  2021-03-24T14:00:00+08:00       4.000000  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...
7    Redis01                969               115  2021-03-24T14:00:00+08:00       1.818182  OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc, ...
8       MG01                822               104  2021-03-24T14:00:00+08:00       3.428571  JVM-Memory_7779_JVM_Memory_HeapMemoryMax, OSLi...
9       MG02                821               106  2021-03-24T14:00:00+08:00       3.260206  JVM-Memory_7779_JVM_Memory_HeapMemoryMax, OSLi...
10      IG01                492                88  2021-03-24T14:00:00+08:00       2.137403  JVM-Memory_7778_JVM_Memory_HeapMemoryMax, JVM-...
11      IG02                469                86  2021-03-24T14:00:00+08:00       1.968354  JVM-Memory_7778_JVM_Memory_HeapMemoryMax, OSLi...
12  apache02                451                82  2021-03-24T14:00:00+08:00       2.009132  OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT, ...
13  apache01                425                80  2021-03-24T14:00:00+08:00       1.688183  OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc, ...```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse variables if present; otherwise load trace_df
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# Components list and timezone (reuse if available)
components = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
              'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, trace_name) using the full series (must be done before time filtering)
trace_p95 = (
    trace_df[trace_df['cmdb_id'].isin(components)]
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(p95_value=('value', lambda s: s.dropna().quantile(0.95)))
)

# 2) Filter trace rows to components and the UTC+8 window 2021-03-24 14:00:00 → 14:30:00 (inclusive)
start_local = pd.Timestamp('2021-03-24 14:00:00').tz_localize(tz)
end_local = pd.Timestamp('2021-03-24 14:30:00').tz_localize(tz)

df_trace = trace_df[trace_df['cmdb_id'].isin(components)].copy()
df_trace['dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
df_window = df_trace[(df_trace['dt'] >= start_local) & (df_trace['dt'] <= end_local)].copy()

# 3) Attach global p95 to windowed rows
df_window = df_window.merge(trace_p95, on=['cmdb_id', 'trace_name'], how='left')

# 4) Aggregate per (cmdb_id, trace_name) to compute anomaly_count, earliest anomaly ts, max_value, p95_value, max_over_p95
def agg_trace_group(g):
    p95 = g['p95_value'].iloc[0]
    if pd.isna(p95):
        anomaly_count = 0
        earliest = pd.NaT
    else:
        anomaly_mask = g['value'] >= p95
        anomaly_count = int(anomaly_mask.sum())
        earliest = g.loc[anomaly_mask, 'dt'].min() if anomaly_count > 0 else pd.NaT
    max_value = g['value'].max() if not g['value'].empty else np.nan
    if pd.isna(p95) or p95 == 0:
        max_over = None
    else:
        max_over = max_value / p95
    return pd.Series({
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest.isoformat() if pd.notnull(earliest) else None,
        'max_value': max_value,
        'p95_value': p95,
        'max_over_p95': max_over
    })

trace_agg = (
    df_window
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .apply(lambda g: agg_trace_group(g))
    .reset_index()
)

# 5) Select requested columns, sort by anomaly_count desc, limit to top 20
trace_top20 = (
    trace_agg[['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','max_value','p95_value','max_over_p95']]
    .sort_values('anomaly_count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

trace_top20

```
Out[5]:
```
```python
# Build a concise plain-English summary from the computed trace_top20 DataFrame
# Reuse trace_top20 from prior computation
top3 = trace_top20.head(3)
unique_counts = sorted(trace_top20['anomaly_count'].unique(), reverse=True)
earliest_vals = sorted(set([ts for ts in trace_top20['earliest_anomaly_timestamp'] if ts]), reverse=False)

examples = []
for i, row in top3.reset_index(drop=True).iterrows():
    examples.append(f"{i+1}. {row['cmdb_id']} / {row['trace_name']} -> anomalies={row['anomaly_count']}, earliest={row['earliest_anomaly_timestamp']}, max={row['max_value']:.6g}, p95={row['p95_value']:.6g}")

summary = (
    "Trace anomaly summary (30-min window, UTC+8):\n"
    f"- Top anomaly counts observed: {unique_counts} (most frequent top counts are 6,5,3,2).\n"
    f"- Earliest anomaly timestamps in the top groups: {earliest_vals} (many peak around 2021-03-24T14:20:00+08:00).\n"
    "- Notable patterns:\n"
    "  * The highest anomaly_count groups (6) are mainly duration_p95 traces hitting ~60s (likely timeouts) on IG02, MG01 and Tomcat04-related flows.\n"
    "  * Several IG nodes (IG01/IG02) show elevated durations and row_count spikes for calls to Tomcat services (max_over_p95 up to ~1.45), indicating slower downstream responses.\n"
    "- Examples (top 3):\n  " + "\n  ".join(examples) + "\n"
    "\nRecommendation:\n"
    "- Investigate the ~14:20 window for potential request timeouts between Tomcat <-> MG/IG components (duration_p95 ≈ 60s).\n"
    "- Prioritize tracing calls listed above (IG02 trace.self.duration_p95; MG01/ Tomcat04 traces) and inspect backend latency/connection issues for the implicated paths."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                        trace_name  anomaly_count earliest_anomaly_timestamp    max_value    p95_value  max_over_p95
0       IG02           trace.self.duration_p95              6  2021-03-24T14:20:00+08:00    60.000000    60.000000      1.000000
1       MG01  trace.from_Tomcat04.duration_p95              6  2021-03-24T14:20:00+08:00    60.001000    60.001000      1.000000
2   Tomcat04        trace.to_MG01.duration_p95              6  2021-03-24T14:20:00+08:00    60.001000    60.001000      1.000000
3       MG02  trace.from_Tomcat04.duration_p95              5  2021-03-24T14:20:00+08:00    60.001000    60.001000      1.000000
4   Tomcat04        trace.to_MG02.duration_p95              5  2021-03-24T14:20:00+08:00    60.001000    60.001000      1.000000
5       MG02  trace.from_Tomcat03.duration_p95              3  2021-03-24T14:20:00+08:00    60.001050    60.001000      1.000001
6   Tomcat03        trace.to_MG02.duration_p95              3  2021-03-24T14:20:00+08:00    60.001050    60.001000      1.000001
7       IG01              trace.self.row_count              2  2021-03-24T14:12:00+08:00  2096.000000  2058.000000      1.018465
8       IG01   trace.to_Tomcat03.duration_mean              2  2021-03-24T14:22:00+08:00   210.534923   148.266402      1.419977
9       IG01    trace.to_Tomcat03.duration_p95              2  2021-03-24T14:21:00+08:00   273.291400   231.214650      1.181981
10      IG01       trace.to_Tomcat03.row_count              2  2021-03-24T14:12:00+08:00   264.000000   254.500000      1.037328
11      IG01          trace.self.duration_mean              2  2021-03-24T14:23:00+08:00    60.000086    56.368981      1.064417
12      IG01    trace.to_Tomcat01.duration_p95              2  2021-03-24T14:21:00+08:00   335.997100   246.688975      1.362027
13      IG01    trace.to_Tomcat02.duration_p95              2  2021-03-24T14:22:00+08:00   239.923200   172.025100      1.394699
14      IG01   trace.to_Tomcat02.duration_mean              2  2021-03-24T14:22:00+08:00   172.672000   117.795929      1.465857
15      IG01   trace.to_Tomcat01.duration_mean              2  2021-03-24T14:21:00+08:00   316.793708   218.041871      1.452903
16      IG02    trace.to_Tomcat02.duration_p95              2  2021-03-24T14:22:00+08:00   235.122750   166.841225      1.409261
17      IG02       trace.to_Tomcat02.row_count              2  2021-03-24T14:12:00+08:00   285.000000   268.000000      1.063433
18      IG02   trace.to_Tomcat03.duration_mean              2  2021-03-24T14:22:00+08:00   200.562259   145.643743      1.377074
19      IG01       trace.to_Tomcat04.row_count              2  2021-03-24T14:12:00+08:00   273.000000   262.500000      1.040000```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Reuse or load DataFrames
try:
    error_df
except NameError:
    error_df = pd.read_csv('error_logs.csv')
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

tz = pytz.timezone('Asia/Shanghai')

# Define UTC+8 time window (inclusive)
start_local = pd.Timestamp('2021-03-24 14:00:00').tz_localize(tz)
end_local = pd.Timestamp('2021-03-24 14:30:00').tz_localize(tz)

# Prepare error_logs: ensure dt column exists and is timezone-aware (UTC->convert to TZ)
error_df['dt'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
error_window = error_df[(error_df['dt'] >= start_local) & (error_df['dt'] <= end_local)].copy()

# 1) Group by cmdb_id: count, earliest, latest (top 10 by count)
error_group = (
    error_window
    .groupby('cmdb_id', as_index=False)
    .agg(
        count=('timestamp', 'size'),
        earliest_ts=('dt', 'min'),
        latest_ts=('dt', 'max')
    )
    .sort_values('count', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

# Format timestamps to ISO-like strings compactly
error_group['earliest_ts'] = error_group['earliest_ts'].astype(str)
error_group['latest_ts'] = error_group['latest_ts'].astype(str)
error_group_top10 = error_group[['cmdb_id','count','earliest_ts','latest_ts']]

# For top 3 cmdb_ids, get up to 5 distinct sample messages each with their earliest message timestamp
top3_cmdbs = error_group_top10['cmdb_id'].head(3).tolist()

msgs = (
    error_window[error_window['cmdb_id'].isin(top3_cmdbs)]
    .groupby(['cmdb_id','message'], as_index=False)
    .agg(earliest_message_ts=('dt', 'min'))
)

# Keep earliest 5 messages per cmdb_id
msgs_sorted = msgs.sort_values(['cmdb_id','earliest_message_ts'])
msgs_top5_per_cmdb = msgs_sorted.groupby('cmdb_id', group_keys=False).head(5).reset_index(drop=True)
# Format timestamp
msgs_top5_per_cmdb['earliest_message_ts'] = msgs_top5_per_cmdb['earliest_message_ts'].astype(str)
messages_top3_df = msgs_top5_per_cmdb[['cmdb_id','message','earliest_message_ts']]

# 2) log.csv: filter window, log_name == 'log.error_count' and value > 0
log_df['dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
log_window = log_df[(log_df['dt'] >= start_local) & (log_df['dt'] <= end_local)].copy()

log_errors = (
    log_window[(log_window['log_name'] == 'log.error_count') & (log_window['value'] > 0)]
    .sort_values('dt')
    .head(50)
    .copy()
)
# Format timestamp for display
log_errors['timestamp_UTC+8'] = log_errors['dt'].astype(str)
log_errors_top50 = log_errors[['timestamp_UTC+8','cmdb_id','log_name','value']].reset_index(drop=True)

# Display compact results
error_group_top10, messages_top3_df, log_errors_top50

```
Out[6]:
```
```python
# Summarize the error_logs and log error_count results in plain English (compact).
# Reuse previously computed variables: error_group_top10, messages_top3_df, log_errors_top50

def _truncate(s, n=100):
    s = str(s)
    return s if len(s) <= n else s[:n-3] + "..."

# Build summary
top_errors_rows = error_group_top10.to_dict(orient='records')
summary_lines = []
summary_lines.append("1) Error logs (2021-03-24 14:00→14:30 UTC+8) — top cmdb_ids by count:")
for r in top_errors_rows:
    summary_lines.append(f" - {r['cmdb_id']}: count={r['count']}, earliest={r['earliest_ts']}, latest={r['latest_ts']}")

# Top-3 sample messages (up to 5 each)
summary_lines.append("\nTop 3 components' sample error messages (up to 5 each):")
for cmdb in top_errors_rows[:3]:
    cid = cmdb['cmdb_id']
    msgs = messages_top3_df[messages_top3_df['cmdb_id'] == cid].head(5)
    summary_lines.append(f" - {cid}:")
    if msgs.empty:
        summary_lines.append("    (no sample messages)")
    else:
        for _, m in msgs.iterrows():
            summary_lines.append(f"    • {m['earliest_message_ts']}  { _truncate(m['message'], 140) }")

# log.error_count hits
log_rows = log_errors_top50.copy()
n_log_rows = len(log_rows)
summary_lines.append(f"\n2) log.error_count > 0 in the same window: {n_log_rows} rows (up to 50 shown).")
# show the most notable high-error timestamps (top 5 by value)
if n_log_rows > 0:
    top_log = log_rows.sort_values('value', ascending=False).head(5)
    summary_lines.append(" Notable log.error_count entries (top 5 by value):")
    for _, lr in top_log.iterrows():
        summary_lines.append(f"  - {lr['timestamp_UTC+8']}  {lr['cmdb_id']}  value={lr['value']}")

summary_lines.append("\nQuick interpretation:")
summary_lines.append(" - Error log activity is concentrated in Tomcat nodes (Tomcat03/04/02/01) and apache nodes within ~14:20–14:26.")
summary_lines.append(" - Top components by error log count: Tomcat03 (31), Tomcat04 (23), Tomcat02 (20), Tomcat01 (20), apache02 (14), apache01 (10).")
summary_lines.append(" - Sample messages show many POST calls to /UOCP/... ServiceTest endpoints originating from IG01/IG02, suggesting frontend requests hitting Tomcat backends.")
summary_lines.append(" - log.error_count shows spikes at 14:26 where multiple Tomcat instances report large error counts (e.g., Tomcat03=24, Tomcat01=20, Tomcat04=20).")
summary_lines.append(" - Recommended next steps: triage Tomcat instances (start with Tomcat03/04/02), inspect the sample POST request errors and correlate with trace/metric spikes around 14:20–14:26.")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id  count                earliest_ts                  latest_ts
0  Tomcat03     31  2021-03-24 14:22:07+08:00  2021-03-24 14:26:36+08:00
1  Tomcat04     23  2021-03-24 14:22:52+08:00  2021-03-24 14:26:35+08:00
2  Tomcat02     20  2021-03-24 14:21:20+08:00  2021-03-24 14:26:35+08:00
3  Tomcat01     20  2021-03-24 14:26:39+08:00  2021-03-24 14:26:50+08:00
4  apache02     14  2021-03-24 14:20:09+08:00  2021-03-24 14:25:20+08:00
5  apache01     10  2021-03-24 14:20:44+08:00  2021-03-24 14:24:54+08:00,      cmdb_id                                            message        earliest_message_ts
0   Tomcat02  IG01 POST /UOCP/person/ServiceTest9.json HTTP/...  2021-03-24 14:21:20+08:00
1   Tomcat02  IG01 POST /UOCP/base/ServiceTest6.json HTTP/1....  2021-03-24 14:21:26+08:00
2   Tomcat02  IG02 POST /UOCP/base/ServiceTest8.json HTTP/1....  2021-03-24 14:21:29+08:00
3   Tomcat02  IG01 POST /UOCP/person/ServiceTest3.json HTTP/...  2021-03-24 14:22:30+08:00
4   Tomcat02  IG02 POST /UOCP/base/ServiceTest2.json HTTP/1....  2021-03-24 14:23:00+08:00
5   Tomcat03  IG02 POST /UOCP/base/ServiceTest4.json HTTP/1....  2021-03-24 14:22:07+08:00
6   Tomcat03  IG02 POST /UOCP/person/ServiceTest10.json HTTP...  2021-03-24 14:22:38+08:00
7   Tomcat03  IG01 POST /UOCP/person/ServiceTest3.json HTTP/...  2021-03-24 14:22:39+08:00
8   Tomcat03  IG01 POST /UOCP/person/ServiceTest5.json HTTP/...  2021-03-24 14:22:39+08:00
9   Tomcat03  IG01 POST /UOCP/person/ServiceTest5.json HTTP/...  2021-03-24 14:22:39+08:00
10  Tomcat04  IG02 POST /UOCP/base/ServiceTest11.json HTTP/1...  2021-03-24 14:22:52+08:00
11  Tomcat04  IG02 POST /UOCP/person/ServiceTest1.json HTTP/...  2021-03-24 14:23:01+08:00
12  Tomcat04  IG01 POST /UOCP/base/ServiceTest11.json HTTP/1...  2021-03-24 14:23:05+08:00
13  Tomcat04  IG01 POST /UOCP/person/ServiceTest10.json HTTP...  2021-03-24 14:26:23+08:00
14  Tomcat04  IG01 POST /UOCP/person/ServiceTest10.json HTTP...  2021-03-24 14:26:24+08:00,               timestamp_UTC+8   cmdb_id         log_name  value
0   2021-03-24 14:20:00+08:00  apache01  log.error_count    1.0
1   2021-03-24 14:20:00+08:00  apache02  log.error_count    1.0
2   2021-03-24 14:21:00+08:00  Tomcat02  log.error_count    3.0
3   2021-03-24 14:21:00+08:00  apache01  log.error_count    1.0
4   2021-03-24 14:21:00+08:00  apache02  log.error_count    2.0
..                        ...       ...              ...    ...
16  2021-03-24 14:25:00+08:00  apache02  log.error_count    2.0
17  2021-03-24 14:26:00+08:00  Tomcat03  log.error_count   24.0
18  2021-03-24 14:26:00+08:00  Tomcat01  log.error_count   20.0
19  2021-03-24 14:26:00+08:00  Tomcat02  log.error_count   15.0
20  2021-03-24 14:26:00+08:00  Tomcat04  log.error_count   20.0

[21 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing DataFrames if present; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
try:
    error_df
except NameError:
    error_df = pd.read_csv('error_logs.csv')
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

tz = pytz.timezone('Asia/Shanghai')

# Time window (UTC+8)
start_local = pd.Timestamp('2021-03-24 14:00:00').tz_localize(tz)
end_local = pd.Timestamp('2021-03-24 14:30:00').tz_localize(tz)

# Ensure datetime columns for trace, error, log (UTC->convert to Asia/Shanghai)
trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
error_df['dt'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
log_df['dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, trace_name) using entire trace series (before filtering)
trace_p95_all = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(p95_value=('value', lambda s: s.dropna().quantile(0.95)))
)

# 2) Filter trace rows to the time window
trace_window = trace_df[(trace_df['dt'] >= start_local) & (trace_df['dt'] <= end_local)].copy()

# Attach global p95 to the windowed trace rows
trace_window = trace_window.merge(trace_p95_all, on=['cmdb_id', 'trace_name'], how='left')

# Tomcats to analyze
tomcats = ['Tomcat01','Tomcat02','Tomcat03','Tomcat04']

rows = []
for tom in tomcats:
    # Condition (a): trace_name contains 'to_Tomcat0X' or 'from_Tomcat0X' (any cmdb_id)
    pattern_to = f"to_{tom}"
    pattern_from = f"from_{tom}"
    cond_a = trace_window['trace_name'].str.contains(pattern_to, na=False) | trace_window['trace_name'].str.contains(pattern_from, na=False)
    # Condition (b): cmdb_id == tomcat AND trace_name starts with 'trace.self'
    cond_b = (trace_window['cmdb_id'] == tom) & trace_window['trace_name'].str.startswith('trace.self', na=False)
    # Anomaly mask: value >= p95_value (p95_value may be NaN -> not anomaly)
    anomaly_mask = (trace_window['value'] >= trace_window['p95_value'])
    selected = trace_window[(cond_a | cond_b) & anomaly_mask].copy()
    if not selected.empty:
        earliest_trace_anomaly = selected['dt'].min().isoformat()
    else:
        earliest_trace_anomaly = None

    # Error logs within window for this tomcat
    err_sel = error_df[(error_df['dt'] >= start_local) & (error_df['dt'] <= end_local) & (error_df['cmdb_id'] == tom)].copy()
    if not err_sel.empty:
        earliest_error_log = err_sel['dt'].min().isoformat()
        error_log_count = int(err_sel.shape[0])
    else:
        earliest_error_log = None
        error_log_count = 0

    # log.error_count > 0 within window for this tomcat
    log_sel = log_df[(log_df['dt'] >= start_local) & (log_df['dt'] <= end_local) & 
                     (log_df['cmdb_id'] == tom) & (log_df['log_name'] == 'log.error_count') & (log_df['value'] > 0)].copy()
    if not log_sel.empty:
        earliest_log_error_count = log_sel['dt'].min().isoformat()
        max_log_error_count = float(log_sel['value'].max())
    else:
        earliest_log_error_count = None
        max_log_error_count = None

    rows.append({
        'tomcat': tom,
        'earliest_trace_anomaly': earliest_trace_anomaly,
        'earliest_error_log': earliest_error_log,
        'error_log_count': error_log_count,
        'earliest_log_error_count': earliest_log_error_count,
        'max_log_error_count': max_log_error_count
    })

result_df = pd.DataFrame(rows, columns=['tomcat','earliest_trace_anomaly','earliest_error_log','error_log_count','earliest_log_error_count','max_log_error_count'])

result_df

```
Out[7]:
```
```python
# Build a concise plain-English summary from the computed result_df (per-Tomcat one-row summary)
lines = []
lines.append("Tomcat summaries (window 2021-03-24 14:00→14:30 UTC+8):")
for _, r in result_df.iterrows():
    tom = r['tomcat']
    ta = r['earliest_trace_anomaly'] or "NULL"
    el = r['earliest_error_log'] or "NULL"
    ec = int(r['error_log_count'])
    le = r['earliest_log_error_count'] or "NULL"
    lm = r['max_log_error_count'] if pd.notnull(r['max_log_error_count']) else "NULL"
    lines.append(f"- {tom}: earliest_trace_anomaly={ta}; earliest_error_log={el} (count={ec}); earliest_log_error_count={le}; max_log_error_count={lm}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

tomcat     earliest_trace_anomaly         earliest_error_log  error_log_count   earliest_log_error_count  max_log_error_count
0  Tomcat01  2021-03-24T14:01:00+08:00  2021-03-24T14:26:39+08:00               20  2021-03-24T14:26:00+08:00                 20.0
1  Tomcat02  2021-03-24T14:12:00+08:00  2021-03-24T14:21:20+08:00               20  2021-03-24T14:21:00+08:00                 15.0
2  Tomcat03  2021-03-24T14:12:00+08:00  2021-03-24T14:22:07+08:00               31  2021-03-24T14:22:00+08:00                 24.0
3  Tomcat04  2021-03-24T14:12:00+08:00  2021-03-24T14:22:52+08:00               23  2021-03-24T14:22:00+08:00                 20.0```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing trace_df and trace_p95_all if available; otherwise load/compute
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure dt column exists (UTC->convert to Asia/Shanghai)
if 'dt' not in trace_df.columns:
    trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure timezone aware and converted
    trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Compute global P95 per (cmdb_id, trace_name) if not already computed
try:
    trace_p95_all
except NameError:
    trace_p95_all = (
        trace_df
        .groupby(['cmdb_id', 'trace_name'], as_index=False)
        .agg(p95_value=('value', lambda s: s.dropna().quantile(0.95)))
    )

# Define window 2021-03-24 14:15:00 -> 2021-03-24 14:24:00 (UTC+8)
start_local = pd.Timestamp('2021-03-24 14:15:00').tz_localize(tz)
end_local = pd.Timestamp('2021-03-24 14:24:00').tz_localize(tz)

# Filter to window
trace_window2 = trace_df[(trace_df['dt'] >= start_local) & (trace_df['dt'] <= end_local)].copy()

# Conditions:
conds_to = trace_window2['trace_name'].str.contains('to_Tomcat02|to_Tomcat03|to_Tomcat04', na=False)
conds_self = (trace_window2['cmdb_id'].isin(['Tomcat02','Tomcat03','Tomcat04'])) & trace_window2['trace_name'].str.startswith('trace.self', na=False)

selected = trace_window2[conds_to | conds_self].copy()

# Attach global p95 values (calculated before time filtering)
selected = selected.merge(trace_p95_all, on=['cmdb_id', 'trace_name'], how='left')

# Determine anomaly mask (value >= p95_value). If p95_value is NaN -> not anomaly
selected['is_anomaly'] = False
mask_valid_p95 = selected['p95_value'].notna()
selected.loc[mask_valid_p95, 'is_anomaly'] = selected.loc[mask_valid_p95, 'value'] >= selected.loc[mask_valid_p95, 'p95_value']

# Aggregate per (cmdb_id, trace_name)
def agg_fn(g):
    anom_count = int(g['is_anomaly'].sum())
    if anom_count > 0:
        earliest = g.loc[g['is_anomaly'], 'dt'].min()
        earliest_iso = earliest.isoformat()
    else:
        earliest_iso = None
    max_val = float(g['value'].max()) if not g['value'].empty else np.nan
    p95 = float(g['p95_value'].iloc[0]) if pd.notna(g['p95_value'].iloc[0]) else np.nan
    if pd.isna(p95) or p95 == 0:
        max_over = None
    else:
        max_over = max_val / p95
    return pd.Series({
        'anomaly_count': anom_count,
        'earliest_anomaly_timestamp': earliest_iso,
        'max_value': max_val,
        'p95_value': p95 if not pd.isna(p95) else None,
        'max_over_p95': max_over
    })

trace_agg = (
    selected
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .apply(lambda g: agg_fn(g))
    .reset_index()
)

# Order by earliest_anomaly_timestamp ascending (None/NULLs go last), limit to top 20
# Convert earliest_anomaly_timestamp to datetime for sorting, keeping None as NaT
trace_agg['earliest_dt_sort'] = pd.to_datetime(trace_agg['earliest_anomaly_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)
trace_event_top20 = (
    trace_agg
    .sort_values(by=['earliest_dt_sort'], ascending=True)
    .head(20)
    [['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','max_value','p95_value','max_over_p95']]
    .reset_index(drop=True)
)

trace_event_top20

```
Out[8]:
```
```python
summary = (
    "Trace anomaly summary (filtered window 14:15→14:24 UTC+8):\n"
    "- Earliest anomalies appear at 2021-03-24T14:16:00+08:00: several row_count spikes targeting Tomcat02/03/04 (seen from IG01/IG02) and trace.self.row_count on Tomcat02/03/04.\n"
    "- Two anomaly types observed:\n"
    "  1) row_count spikes (IG → Tomcat and Tomcat self): single-minute hits at ~14:16 (max_over_p95 ≈ 1.01–1.08).\n"
    "  2) latency/duration spikes: starts ~14:20–14:22 with multiple IG→Tomcat duration_p95/mean traces exceeding global P95 (max_over_p95 typically 1.18–1.40).\n"
    "- Highest relative severity in this subset: Tomcat02 trace.self.duration_mean (anomaly at 14:20) with max_over_p95 ≈ 2.8057; Tomcat03 trace.self.duration_p95 at ~14:22 shows max_over_p95 ≈ 2.2583 — these indicate per-Tomcat processing durations rose multiple times above their historical P95.\n"
    "- Many IG→Tomcat paths (IG01/IG02 to Tomcat02/03/04) show elevated durations around 14:21–14:22 (max_over_p95 ~1.32–1.40), suggesting downstream slowness on Tomcat nodes or network/processing delays along those paths.\n"
    "Conclusion / recommended next steps:\n"
    "- Prioritize investigation of Tomcat02 and Tomcat03 (highest relative duration spikes) and the IG→Tomcat request paths around 14:16–14:22.\n"
    "- Correlate these trace anomalies with the Tomcat metric and log error spikes at ~14:20–14:26 to identify root cause (thread/CPU/JVM or backend DB latency).\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                       trace_name  anomaly_count earliest_anomaly_timestamp    max_value    p95_value  max_over_p95
0       IG02      trace.to_Tomcat03.row_count            1.0  2021-03-24T14:16:00+08:00   295.000000   274.500000      1.074681
1   Tomcat03             trace.self.row_count            1.0  2021-03-24T14:16:00+08:00  3608.000000  3461.500000      1.042323
2   Tomcat02             trace.self.row_count            1.0  2021-03-24T14:16:00+08:00  3581.000000  3575.500000      1.001538
3       IG02      trace.to_Tomcat04.row_count            1.0  2021-03-24T14:16:00+08:00   270.000000   266.850000      1.011804
4   Tomcat04             trace.self.row_count            1.0  2021-03-24T14:16:00+08:00  3590.000000  3491.500000      1.028211
5       IG01      trace.to_Tomcat02.row_count            1.0  2021-03-24T14:16:00+08:00   269.000000   252.500000      1.065347
6       IG02      trace.to_Tomcat02.row_count            1.0  2021-03-24T14:18:00+08:00   285.000000   268.000000      1.063433
7   Tomcat02         trace.self.duration_mean            2.0  2021-03-24T14:20:00+08:00     8.047313     2.868219      2.805683
8       IG02   trace.to_Tomcat04.duration_p95            2.0  2021-03-24T14:21:00+08:00   301.841500   219.737790      1.373644
9       IG01   trace.to_Tomcat04.duration_p95            2.0  2021-03-24T14:21:00+08:00   304.946000   217.854650      1.399768
10      IG02   trace.to_Tomcat03.duration_p95            2.0  2021-03-24T14:21:00+08:00   259.980400   220.818875      1.177347
11      IG01   trace.to_Tomcat03.duration_p95            2.0  2021-03-24T14:21:00+08:00   273.291400   231.214650      1.181981
12  Tomcat04         trace.self.duration_mean            2.0  2021-03-24T14:22:00+08:00     3.010880     2.290851      1.314306
13      IG01   trace.to_Tomcat02.duration_p95            2.0  2021-03-24T14:22:00+08:00   239.923200   172.025100      1.394699
14  Tomcat03          trace.self.duration_p95            2.0  2021-03-24T14:22:00+08:00    31.802000    14.082325      2.258292
15  Tomcat03         trace.self.duration_mean            2.0  2021-03-24T14:22:00+08:00     4.305021     3.269534      1.316708
16  Tomcat02          trace.self.duration_p95            2.0  2021-03-24T14:22:00+08:00    30.111750    17.584450      1.712408
17      IG01  trace.to_Tomcat04.duration_mean            2.0  2021-03-24T14:22:00+08:00   174.469114   131.780851      1.323934
18      IG02  trace.to_Tomcat04.duration_mean            2.0  2021-03-24T14:22:00+08:00   171.759000   128.101329      1.340806
19      IG02  trace.to_Tomcat03.duration_mean            2.0  2021-03-24T14:22:00+08:00   200.562259   145.643743      1.377074```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing metric_df and tz if present; otherwise load metric.csv and set timezone
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure a timezone-aware datetime column (UTC -> convert to Asia/Shanghai)
metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Target component and time window
target = 'Tomcat02'
start_local = pd.Timestamp('2021-03-24 14:15:00').tz_localize(tz)
end_local = pd.Timestamp('2021-03-24 14:24:00').tz_localize(tz)

# Rule 9: compute global P95 per (cmdb_id, kpi_name) using the full series BEFORE filtering
p95_full = (
    metric_df[metric_df['cmdb_id'] == target]
    .groupby('kpi_name', as_index=False)
    .agg(p95_value=('value', lambda s: s.dropna().quantile(0.95)))
)

# Filter metric rows to the time window for the target Tomcat02
window_df = metric_df[(metric_df['cmdb_id'] == target) & (metric_df['dt'] >= start_local) & (metric_df['dt'] <= end_local)].copy()

# Attach the global p95 to the windowed rows
window_df = window_df.merge(p95_full, on='kpi_name', how='left')

# Determine anomaly (value >= p95) only where p95 is valid
mask_valid = window_df['p95_value'].notna()
window_df['is_anomaly'] = False
window_df.loc[mask_valid, 'is_anomaly'] = window_df.loc[mask_valid, 'value'] >= window_df.loc[mask_valid, 'p95_value']

# Aggregate per kpi_name
def agg_kpi(g):
    p95 = g['p95_value'].iloc[0]
    anom_count = int(g['is_anomaly'].sum())
    earliest = g.loc[g['is_anomaly'], 'dt'].min() if anom_count > 0 else pd.NaT
    earliest_iso = earliest.isoformat() if pd.notnull(earliest) else None
    max_val = float(g['value'].max()) if not g['value'].empty else np.nan
    # max_over_p95 handling
    if pd.isna(p95) or p95 == 0:
        max_over = None
    else:
        max_over = max_val / float(p95)
    return pd.Series({
        'anomaly_count': anom_count,
        'earliest_anomaly_timestamp': earliest_iso,
        'max_value': max_val,
        'p95_value': float(p95) if pd.notna(p95) else None,
        'max_over_p95': max_over
    })

kpi_agg = (
    window_df
    .groupby('kpi_name', as_index=False)
    .apply(lambda g: agg_kpi(g))
    .reset_index()
)

# Sort by max_over_p95 descending, treating None as smallest
kpi_agg['max_over_sort'] = kpi_agg['max_over_p95'].apply(lambda x: x if x is not None else -np.inf)
top10 = (
    kpi_agg
    .sort_values('max_over_sort', ascending=False)
    .head(10)
    [['kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value','p95_value','max_over_p95']]
    .reset_index(drop=True)
)

top10

```
Out[9]:
```
```python
# Build a concise plain-English summary for Tomcat02 KPI anomalies using the previously computed `top10` table.
# Reuse existing 'top10' DataFrame from prior computation.

def _short(s, n=120):
    s = str(s)
    return s if len(s) <= n else s[:n-3] + "..."

lines = []
lines.append("Tomcat02 metric anomaly summary (window 2021-03-24 14:15→14:24 UTC+8):")
lines.append("- Global P95 values were computed from the full series before filtering; anomalies are value >= P95 within the 30-min window subset.")
lines.append(f"- Top {min(10, len(top10))} KPIs ranked by max_over_p95 (desc):")

for i, r in top10.iterrows():
    k = _short(r['kpi_name'], 120)
    ac = int(r['anomaly_count']) if pd.notnull(r['anomaly_count']) else 0
    et = r['earliest_anomaly_timestamp'] if r['earliest_anomaly_timestamp'] else "NULL"
    mv = f"{r['max_value']:.6g}" if pd.notnull(r['max_value']) else "NULL"
    p95 = f"{r['p95_value']:.6g}" if pd.notnull(r['p95_value']) else "NULL"
    mo = f"{r['max_over_p95']:.6g}" if (r['max_over_p95'] is not None and not pd.isna(r['max_over_p95'])) else "NULL"
    lines.append(f"  {i+1}. {k} | anomaly_count={ac} | earliest={et} | max={mv} | p95={p95} | max_over_p95={mo}")

lines.append("\nKey observations:")
lines.append("- The largest relative spike is in Tomcat thread count metric (CurrentThreads...), with max_over_p95 ≈ 1.36 at 14:24, indicating thread usage exceeded its historical P95.")
lines.append("- Other notable anomalies (all single-minute hits) include local disk IO (sdb DSKBps), network bandwidth/util, CPU single-core util, and JVM memory metrics—each exceeding their P95 by modest amounts (≈1.01–1.14).")
lines.append("- Most anomalies have anomaly_count=1, i.e., brief spikes rather than sustained exceedance in this 9-minute window.")
lines.append("\nSuggested next steps:")
lines.append("- Inspect Tomcat thread metrics at ~14:24 to see if thread growth correlates with increased request latency or errors.")
lines.append("- Correlate these metric spikes with trace and log anomalies (14:16–14:26 window) and with Tomcat error_count spikes to find root cause (e.g., blocking threads, backend DB latency, or IO contention).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name  anomaly_count earliest_anomaly_timestamp     max_value     p95_value  max_over_p95
0  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...            1.0  2021-03-24T14:24:00+08:00  1.860000e+02  1.370000e+02      1.357664
1     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps            1.0  2021-03-24T14:24:00+08:00  6.000000e+00  5.250000e+00      1.142857
2    OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil            1.0  2021-03-24T14:18:00+08:00  5.800000e-03  5.480000e-03      1.058394
3                    OSLinux-CPU_CPU-0_SingleCpuUtil            1.0  2021-03-24T14:19:00+08:00  5.964700e+00  5.848500e+00      1.019868
4            Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory            1.0  2021-03-24T14:16:00+08:00  2.167614e+09  2.136592e+09      1.014519
5                    OSLinux-CPU_CPU-2_SingleCpuidle            1.0  2021-03-24T14:22:00+08:00  9.858620e+01  9.770970e+01      1.008970
6     Tomcat-MEMORY_7441-MEMORY_JVMMemoryUsedPercent            1.0  2021-03-24T14:18:00+08:00  5.100000e+01  5.070000e+01      1.005917
7    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec            1.0  2021-03-24T14:18:00+08:00  6.336140e+05  6.303912e+05      1.005112
8                    OSLinux-CPU_CPU-0_SingleCpuidle            1.0  2021-03-24T14:20:00+08:00  9.740562e+01  9.696827e+01      1.004510
9  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...            1.0  2021-03-24T14:19:00+08:00  2.290000e+02  2.280000e+02      1.004386```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

# Reuse metric_df and tz if available; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure timezone-aware datetime column (UTC -> convert to Asia/Shanghai)
metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Target component and full-series P95 calculation (Rule 9: compute BEFORE time filtering)
target = 'Tomcat02'
p95_tomcat02 = (
    metric_df[metric_df['cmdb_id'] == target]
    .groupby('kpi_name', as_index=False)
    .agg(p95_value=('value', lambda s: s.dropna().quantile(0.95)))
)

# Time window (inclusive) 2021-03-24 14:15:00 → 2021-03-24 14:26:59 (UTC+8)
start_local = pd.Timestamp('2021-03-24 14:15:00').tz_localize(tz)
end_local = pd.Timestamp('2021-03-24 14:26:59').tz_localize(tz)

# Desired minute timestamps (UTC+8) to include
minute_list = [
    pd.Timestamp('2021-03-24 14:16:00').tz_localize(tz),
    pd.Timestamp('2021-03-24 14:20:00').tz_localize(tz),
    pd.Timestamp('2021-03-24 14:21:00').tz_localize(tz),
    pd.Timestamp('2021-03-24 14:22:00').tz_localize(tz),
    pd.Timestamp('2021-03-24 14:24:00').tz_localize(tz),
    pd.Timestamp('2021-03-24 14:26:00').tz_localize(tz)
]
minute_set = set(minute_list)

# KPI substring patterns to match
patterns = [
    'Tomcat-Requests',
    'Tomcat-Threads',
    'Tomcat-MEMORY',
    'OSLinux-CPU',
    'OSLinux-OSLinux_LOCALDISK'
]
# Create a regex OR pattern for contains matching (case-sensitive as in data)
regex_pattern = '|'.join([p for p in patterns])

# Filter metric rows for Tomcat02 within window and matching kpi_name patterns
window_sel = metric_df[
    (metric_df['cmdb_id'] == target) &
    (metric_df['dt'] >= start_local) &
    (metric_df['dt'] <= end_local) &
    (metric_df['kpi_name'].str.contains(regex_pattern, na=False))
].copy()

# Keep only rows whose minute (floor to minute) is one of the specified minutes
window_sel['dt_min'] = window_sel['dt'].dt.floor('T')
window_sel = window_sel[window_sel['dt_min'].isin(minute_set)].copy()

# Attach global p95 values computed earlier
window_sel = window_sel.merge(p95_tomcat02, on='kpi_name', how='left')

# Determine breached boolean: value >= p95_value (only when p95_value is not null)
window_sel['breached'] = False
mask_p95_valid = window_sel['p95_value'].notna()
window_sel.loc[mask_p95_valid, 'breached'] = window_sel.loc[mask_p95_valid, 'value'] >= window_sel.loc[mask_p95_valid, 'p95_value']

# Prepare output columns: timestamp (UTC+8 ISO), kpi_name, value, p95_value, breached
window_sel['timestamp_UTC+8'] = window_sel['dt'].dt.tz_convert(tz).astype(str)

result_rows = (
    window_sel[['timestamp_UTC+8','kpi_name','value','p95_value','breached']]
    .sort_values(['timestamp_UTC+8','kpi_name'], ascending=[True, True])
    .head(50)
    .reset_index(drop=True)
)

result_rows

```
Out[10]:
```
```python
# Build a concise plain-English summary from the previously computed `result_rows` DataFrame.
# Reuse result_rows from earlier state (filtered Tomcat02 metrics at selected minutes).

import pandas as pd
import numpy as np

# If result_rows is not present, load minimal fallback (should not be needed in stateful kernel)
try:
    result_rows
except NameError:
    result_rows = pd.read_csv('metric.csv')  # fallback (won't be used meaningfully)

total_rows = len(result_rows)
breached_df = result_rows[result_rows['breached'] == True].copy()
n_breached = len(breached_df)
unique_breached_kpis = breached_df['kpi_name'].dropna().unique().tolist()[:10]
earliest_breach_ts = breached_df['timestamp_UTC+8'].min() if n_breached > 0 else None

# For each breached KPI, compute earliest breach timestamp and a sample of value vs p95 ratio
kpi_summary_rows = []
for kpi, g in breached_df.groupby('kpi_name'):
    earliest = g['timestamp_UTC+8'].min()
    max_val = g['value'].max()
    p95 = g['p95_value'].iloc[0]
    if pd.isna(p95) or p95 == 0:
        ratio = None
    else:
        ratio = float(max_val) / float(p95)
    kpi_summary_rows.append((kpi, earliest, max_val, p95, ratio))

# Keep up to 8 example breached KPIs
kpi_summary_rows = sorted(kpi_summary_rows, key=lambda x: (x[1] or ""), reverse=False)[:8]

# Compose summary string
lines = []
lines.append("Tomcat02 — selected metric rows summary (minutes filtered: 14:16,14:20,14:21,14:22,14:24,14:26 UTC+8):")
lines.append(f"- Total matching metric rows returned: {total_rows}")
lines.append(f"- Number of rows breaching their global P95 (value >= P95): {n_breached}")
if n_breached > 0:
    lines.append(f"- Earliest breach observed at: {earliest_breach_ts}")
    lines.append("- Breached KPI examples (up to 8):")
    for kpi, earliest, max_val, p95, ratio in kpi_summary_rows:
        ratio_str = f"{ratio:.3f}x" if ratio is not None else "NULL (p95=0 or missing)"
        lines.append(f"  • {kpi} | earliest_breach={earliest} | max_value={max_val:.6g} | p95={p95 if not pd.isna(p95) else 'NULL'} | max_over_p95={ratio_str}")
else:
    lines.append("- No breaches detected among the selected rows.")

# Observations and recommendations
lines.append("\nObservations:")
lines.append("- Most selected rows did NOT breach their global P95; breaches are relatively few.")
lines.append("- One notable case: some disk read/transfer metrics have p95==0 in the historical series; those appear as 'breached' when current value >= 0 — treat such cases with caution because ratio is undefined.")
lines.append("\nRecommended next steps:")
lines.append("- Investigate the listed breached KPIs and their timestamps (above) to correlate with trace and log anomalies around 14:16–14:26.")
lines.append("- Pay attention to Tomcat thread and JVM memory metrics if they appear among breaches (these can indicate resource contention).")
lines.append("- For metrics with p95==0, examine historical data and consider a small nonzero floor for ratio calculations or use absolute thresholds instead.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

timestamp_UTC+8                                           kpi_name      value  p95_value  breached
0   2021-03-24 14:16:00+08:00                    OSLinux-CPU_CPU-0_SingleCpuidle  94.500000  96.968274     False
1   2021-03-24 14:16:00+08:00                    OSLinux-CPU_CPU-1_SingleCpuidle  96.686100  97.885474     False
2   2021-03-24 14:16:00+08:00                    OSLinux-CPU_CPU-2_SingleCpuidle  96.868200  97.709695     False
3   2021-03-24 14:16:00+08:00                    OSLinux-CPU_CPU-3_SingleCpuidle   0.000000  63.374252     False
4   2021-03-24 14:16:00+08:00                         OSLinux-CPU_CPU_CPUCpuUtil  27.688500  28.230360     False
..                        ...                                                ...        ...        ...       ...
45  2021-03-24 14:20:00+08:00    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps   0.733333   1.404175     False
46  2021-03-24 14:20:00+08:00   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite  10.266667  19.133325     False
47  2021-03-24 14:20:00+08:00     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps   0.000000   5.250000     False
48  2021-03-24 14:20:00+08:00  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...   0.000000   0.200000     False
49  2021-03-24 14:20:00+08:00    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps   0.000000   0.000000      True

[50 rows x 5 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import pytz

# Reuse metric_df and tz if present; otherwise load and set timezone
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
tz = pytz.timezone('Asia/Shanghai')

# Ensure timezone-aware datetime column (UTC -> convert to Asia/Shanghai)
metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Rule 9: compute global P95 per (Tomcat02, kpi_name) using full series BEFORE time filtering
target = 'Tomcat02'
p95_full_tomcat02 = (
    metric_df[metric_df['cmdb_id'] == target]
    .groupby('kpi_name', as_index=False)
    .agg(p95_value=('value', lambda s: s.dropna().quantile(0.95)))
)

# KPI substring patterns
patterns = [
    'Tomcat-Requests',
    'Tomcat-Threads',
    'Tomcat-MEMORY',
    'OSLinux-CPU',
    'OSLinux-OSLinux_LOCALDISK',
    'OSLinux-OSLinux_NETWORK'
]
regex_pattern = '|'.join(patterns)

# Time window: 2021-03-24 14:00:00 -> 2021-03-24 14:30:00 (UTC+8), inclusive
start_local = pd.Timestamp('2021-03-24 14:00:00').tz_localize(tz)
end_local = pd.Timestamp('2021-03-24 14:30:00').tz_localize(tz)

# Filter metric rows for Tomcat02, matching KPI patterns, within the window
df_win = metric_df[
    (metric_df['cmdb_id'] == target) &
    (metric_df['dt'] >= start_local) &
    (metric_df['dt'] <= end_local) &
    (metric_df['kpi_name'].str.contains(regex_pattern, na=False))
].copy()

# Attach global p95 values
df_win = df_win.merge(p95_full_tomcat02, on='kpi_name', how='left')

# Prepare container for segments
segments = []

# For each kpi_name, find anomaly rows and consecutive segments
for kpi, grp in df_win.groupby('kpi_name', sort=True):
    grp = grp.sort_values('dt').reset_index(drop=True)
    # If p95 is NaN, skip anomaly detection (cannot compare)
    p95_val = grp['p95_value'].iloc[0]
    if pd.isna(p95_val):
        continue
    # Determine anomaly rows: value >= p95 (per instruction)
    grp['is_anomaly'] = grp['value'] >= p95_val

    # Keep only anomaly rows for segmentation
    anom = grp[grp['is_anomaly']].copy()
    if anom.empty:
        continue

    # Identify consecutive segments by checking time diffs between successive anomaly rows
    anom = anom.sort_values('dt').reset_index(drop=True)
    if anom.shape[0] == 0:
        continue

    # Compute gap from previous anomaly row in minutes
    anom['prev_dt'] = anom['dt'].shift(1)
    anom['gap_minutes'] = (anom['dt'] - anom['prev_dt']).dt.total_seconds().div(60).fillna(0)
    # Start a new segment when gap > 1 minute
    anom['new_segment'] = (anom['gap_minutes'] > 1).astype(int)
    anom['segment_id'] = anom['new_segment'].cumsum()

    # Aggregate per segment
    seg_aggs = anom.groupby('segment_id').agg(
        segment_start_dt=('dt', 'min'),
        segment_end_dt=('dt', 'max'),
        segment_length_minutes=('dt', 'count'),  # number of consecutive anomaly minutes
        max_value_in_segment=('value', 'max')
    ).reset_index(drop=False)

    # Compute max_over_p95 and filter by criteria
    for _, s in seg_aggs.iterrows():
        max_val = float(s['max_value_in_segment'])
        # handle p95 == 0
        if p95_val == 0 or pd.isna(p95_val):
            max_over = None
        else:
            max_over = max_val / float(p95_val)
        length = int(s['segment_length_minutes'])
        # Keep only segments where length >=2 OR max_over_p95 >= 1.5
        keep = (length >= 2) or (max_over is not None and max_over >= 1.5)
        if keep:
            segments.append({
                'kpi_name': kpi,
                'segment_start_timestamp': s['segment_start_dt'].isoformat(),
                'segment_end_timestamp': s['segment_end_dt'].isoformat(),
                'segment_length_minutes': length,
                'max_value_in_segment': max_val,
                'p95_value': float(p95_val) if not pd.isna(p95_val) else None,
                'max_over_p95': max_over
            })

# Create DataFrame of segments, order by segment_start_timestamp asc, limit top 20
segments_df = pd.DataFrame(segments)
if not segments_df.empty:
    segments_df['segment_start_dt_sort'] = pd.to_datetime(segments_df['segment_start_timestamp'], utc=True).dt.tz_convert(tz)
    segments_df = segments_df.sort_values('segment_start_dt_sort', ascending=True).head(20).reset_index(drop=True)
    segments_df = segments_df[['kpi_name','segment_start_timestamp','segment_end_timestamp','segment_length_minutes','max_value_in_segment','p95_value','max_over_p95']]
else:
    # empty result with requested columns
    segments_df = pd.DataFrame(columns=['kpi_name','segment_start_timestamp','segment_end_timestamp','segment_length_minutes','max_value_in_segment','p95_value','max_over_p95'])

# Display compact result (top segments)
segments_df

```
Out[11]:
```
```python
summary = (
    "Tomcat02 anomaly segments (2021-03-24 14:00→14:30 UTC+8) — concise findings:\n\n"
    "Key anomaly segments detected (filtered by length>=2 OR max_over_p95>=1.5):\n"
    "- Tomcat-Threads (CurrentThreads for http-nio-8003): start=2021-03-24T14:24:00+08:00, length=2 min, "
    "max=273, p95=137, max_over_p95≈1.99 — nearly doubled thread count.\n"
    "- OSLinux NETWORK (TCP-CLOSE-WAIT): start=2021-03-24T14:25:00+08:00, length=2 min, "
    "max=181, p95≈93.8, max_over_p95≈1.93 — many sockets in CLOSE_WAIT.\n"
    "- OSLinux-CPU CPU-3 SingleCpuidle: short spike at 2021-03-24T14:00:00+08:00 with max_over_p95≈1.53 (single-minute but notable).\n"
    "- Disk I/O on sda near window end (14:29–14:30): DSKRTps and DSKRead show high ratios (max_over_p95 ≈1.85 and ≈2.42 respectively) — elevated read activity.\n"
    "- Several Tomcat request/thread metrics form long segments spanning the whole window (14:00→14:30) but with max_over_p95 ≈1.0 (values equal to their P95) — sustained high baseline rather than extreme spike.\n"
    "- One network metric (TCP-FIN-WAIT) shows a 31-minute segment but has p95==0 (ratio undefined); treat p95==0 cases cautiously as they can produce misleading flags.\n\n"
    "Interpretation & recommended next steps:\n"
    "1) Prioritize investigation of the Tomcat thread surge (~14:24) and the TCP CLOSE_WAIT spike (~14:25) — these can cause request queuing/errors.\n"
    "2) Correlate these timestamps with trace anomalies and Tomcat error_count/log spikes (14:16–14:26) to confirm impact and root cause.\n"
    "3) For CLOSE_WAIT: check socket states, upstream/backend connection handling, and thread pool exhaustion.\n"
    "4) For disk I/O spikes at ~14:29–14:30, check disk health and backend DB/storage access patterns.\n"
    "5) Re-evaluate metrics with p95==0 before treating them as definitive anomalies.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name    segment_start_timestamp      segment_end_timestamp  segment_length_minutes  max_value_in_segment     p95_value  max_over_p95
0                     OSLinux-CPU_CPU-3_SingleCpuidle  2021-03-24T14:00:00+08:00  2021-03-24T14:00:00+08:00                       1          9.714001e+01  6.337425e+01      1.532799
1        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00                      31          1.000000e+00  0.000000e+00           NaN
2   Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00                      31          3.906190e+05  3.906190e+05      1.000000
3   Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  2021-03-24T14:00:00+08:00  2021-03-24T14:30:00+08:00                      31          5.000000e+02  5.000000e+02      1.000000
4   Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  2021-03-24T14:24:00+08:00  2021-03-24T14:25:00+08:00                       2          2.730000e+02  1.370000e+02      1.992701
5      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  2021-03-24T14:25:00+08:00  2021-03-24T14:26:00+08:00                       2          1.810000e+02  9.380000e+01      1.929638
6   Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  2021-03-24T14:26:00+08:00  2021-03-24T14:28:00+08:00                       3          3.510000e+02  3.510000e+02      1.000000
7                          OSLinux-CPU_CPU_CPUSysTime  2021-03-24T14:27:00+08:00  2021-03-24T14:28:00+08:00                       2          1.864200e+00  1.066143e+00      1.748547
8                         OSLinux-CPU_CPU_CPUUserTime  2021-03-24T14:27:00+08:00  2021-03-24T14:28:00+08:00                       2          2.676430e+01  2.652388e+01      1.009064
9   Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  2021-03-24T14:27:00+08:00  2021-03-24T14:30:00+08:00                       4          7.730000e+02  7.730000e+02      1.000000
10     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps  2021-03-24T14:29:00+08:00  2021-03-24T14:29:00+08:00                       1          1.000000e+00  5.500000e-01      1.818182
11  Tomcat-Requests_7441-"http-nio-8003"_Processin...  2021-03-24T14:29:00+08:00  2021-03-24T14:30:00+08:00                       2          1.884909e+09  1.884235e+09      1.000358
12  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  2021-03-24T14:29:00+08:00  2021-03-24T14:30:00+08:00                       2          3.063285e+06  3.062614e+06      1.000219
13    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps  2021-03-24T14:30:00+08:00  2021-03-24T14:30:00+08:00                       1          5.167000e-01  2.791750e-01      1.850810
14    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead  2021-03-24T14:30:00+08:00  2021-03-24T14:30:00+08:00                       1          2.746670e+01  1.136668e+01      2.416423```
```